<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>PyTorch 핵심 구성요소(Core Building Blocks)</strong>에 대해 학습합니다.</br></br>
>nn.Module, 레이어, 활성화 함수, 정규화 기법으로 신경망을 학습해봅시다.

</br>

# nn.Module
> PyTorch에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">모든 신경망의 기본 클래스</mark>입니다.
> 모델을 정의할 때 반드시 `nn.Module`을 상속합니다.

>파라미터(가중치, 편향)를 리스트나 딕셔너리로 직접 관리하면, 레이어가 늘어날수록 파라미터 추적·저장·불러오기 코드가 기하급수적으로 복잡해지고, `loss.backward()` 후 모든 파라미터를 일일이 업데이트하는 코드도 수동으로 작성해야 합니다.</br>
> PyTorch의 추상화는 이 문제를 해결합니다. </br></br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">nn.Module</mark>은 파라미터 자동 등록과 `state_dict()` 직렬화를 제공하고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">손실 함수</mark>는 검증된 수학적 구현체를 재사용하게 하며, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">옵티마이저</mark>는 `optimizer.step()` 한 줄로 파라미터 업데이트를 처리합니다.</br>
 > 추상화의 목적은 코드 복잡성 제거가 아니라 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">반복 작업을 숨겨 모델 설계에 집중</mark>하게 하는 것입니다.

In [1]:
# TODO 1: torch와 torch.nn을 import 하고 torch.nn은 nn으로 별칭을 부여해봅시다.
import torch
import torch.nn as nn

💡super().__init__()이 필수인 이유
> `nn.Module`을 상속할 때 `super().__init__()`을 호출하면, 내부적으로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">파라미터 저장소와 서브모듈 딕셔너리</mark>가 초기화됩니다.</br>
> 이 호출이 없으면 `self.layer1 = nn.Linear(...)`을 선언해도 `model.parameters()`에 등록되지 않아, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습 대상 파라미터가 0개</mark>가 됩니다.</br></br>
> 즉, `super().__init__()`은 "나는 nn.Module의 기능(파라미터 관리, `state_dict()`, `.to(device)` 등)을 사용하겠다"는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">등록 선언</mark>입니다.

In [8]:
# TODO 2: nn.Module을 상속받는 MyModel을 정의해봅시다.

class MyModel(nn.Module):

    def __init__(self):
        super().__init__()

        # TODO 2-1: layer1에 784 x 256 차원 레이어를 선언해봅시다.
        self.layer1 = nn.Linear(784, 256)
        
        # TODO 2-2: relu에 활성화 함수를 선언해봅시다.
        self.relu = nn.ReLU()
        
        # TODO 2-3: layer2에 256 x 10 차원 레이어를 선언해봅시다.
        self.layer2 = nn.Linear(256, 10)
        
    def forward(self, x):
        # TODO 2-4: layer1의 결과를 ReLU 함수로 전달해봅시다.
        x = self.relu(self.layer1(x))
        
        # TODO 2-5: 그 결과를 layer2로 전달해봅시다.
        x = self.relu(self.layer2(x))
        
        # TODO 2-6: 결과를 반환해봅시다.
        return x

In [11]:
# TODO 3: MyModel 클래스를 인스턴스화하여 my_model에 저장해봅시다.
my_model = MyModel()
print("--- 모델 구조 확인 ---")
print(my_model)

--- 모델 구조 확인 ---
MyModel(
  (layer1): Linear(in_features=784, out_features=256, bias=True)
  (relu): ReLU()
  (layer2): Linear(in_features=256, out_features=10, bias=True)
)


In [12]:
# TODO 4: 모델의 모든 파라미터 개수(학습해야 할 가중치의 개수)를 합산하여 총 파라미터 수를 출력해봅시다.
total_params = sum(param.numel() for param in my_model.parameters())
print(f"\n--- 총 파라미터 수: {total_params:,}개 ---")


--- [TODO 4] 총 파라미터 수: 203,530개 ---


</br>

## 주요 레이어

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">레이어</th>
      <th style="text-align:center">설명</th>
      <th style="text-align:center">예시</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>nn.Linear(in, out)</code></td><td style="text-align:center">완전 연결층</td><td style="text-align:center"><code>nn.Linear(784, 256)</code></td></tr>
    <tr><td style="text-align:center"><code>nn.Conv2d(in, out, k)</code></td><td style="text-align:center">2D 합성곱</td><td style="text-align:center"><code>nn.Conv2d(3, 64, 3)</code></td></tr>
    <tr><td style="text-align:center"><code>nn.BatchNorm1d(n)</code></td><td style="text-align:center">배치 정규화</td><td style="text-align:center"><code>nn.BatchNorm1d(256)</code></td></tr>
    <tr><td style="text-align:center"><code>nn.Dropout(p)</code></td><td style="text-align:center">드롭아웃</td><td style="text-align:center"><code>nn.Dropout(0.5)</code></td></tr>
  </tbody>
</table>

</br>

## 활성화 함수 (Activation Functions)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">비선형 변환</mark>을 적용하여 신경망이 복잡한 패턴을 학습하게 합니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">함수</th>
      <th style="text-align:center">수식</th>
      <th style="text-align:center">특징</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">ReLU</td><td style="text-align:center">$f(x) = \max(0, x)$</td><td style="text-align:center">가장 많이 사용, 빠름</td></tr>
    <tr><td style="text-align:center">Sigmoid</td><td style="text-align:center">$f(x) = \frac{1}{1+e^{-x}}$</td><td style="text-align:center">출력 [0, 1], 이진 분류</td></tr>
    <tr><td style="text-align:center">Tanh</td><td style="text-align:center">$f(x) = \tanh(x)$</td><td style="text-align:center">출력 [-1, 1]</td></tr>
    <tr><td style="text-align:center">Softmax</td><td style="text-align:center">$f(x_i) = \frac{e^{x_i}}{\sum e^{x_j}}$</td><td style="text-align:center">다중 분류 출력</td></tr>
  </tbody>
</table>

In [13]:
# TODO 5: x에 [-2.0, -1.0, 0.0, 1.0, 2.0] 텐서를 생성하고, ReLU, Sigmoid, Tanh 활성화 함수를 각각 적용하여 결과를 출력해봅시다.
x_act = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

# ReLU: 음수는 0으로, 양수는 그대로 통과시킵니다. (가장 널리 쓰임)
relu_result = torch.relu(x_act)

# Sigmoid: 모든 값을 0과 1 사이의 확률값처럼 압축합니다.
sigmoid_result = torch.sigmoid(x_act)

# Tanh: 모든 값을 -1과 1 사이로 압축하며, 중심이 0이라 Sigmoid보다 학습이 잘 됩니다.
tanh_result = torch.tanh(x_act)

print("--- 활성화 함수 적용 결과 ---")
print(f"원본 입력: {x_act}")
print(f"ReLU 적용: {relu_result}")
print(f"Sigmoid 적용: {sigmoid_result}")
print(f"Tanh 적용:   {tanh_result}")

--- 활성화 함수 적용 결과 ---
원본 입력: tensor([-2., -1.,  0.,  1.,  2.])
ReLU 적용: tensor([0., 0., 0., 1., 2.])
Sigmoid 적용: tensor([0.1192, 0.2689, 0.5000, 0.7311, 0.8808])
Tanh 적용:   tensor([-0.9640, -0.7616,  0.0000,  0.7616,  0.9640])


💡ReLU가 기본 선택인 이유
> 기울기 소실(Vanishing Gradient) 문제가 적고 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">계산이 빠릅니다</mark>.</br>
> 특별한 이유가 없다면 ReLU로 시작하세요.

<div style="text-align:center">

</div>

</br>

## Dropout (드롭아웃)
> 학습 중 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">뉴런을 랜덤으로 비활성화</mark>하여 과적합을 방지합니다.

💡train vs eval 모드
> `model.train()`: Dropout 활성화 (학습 시)</br>
> `model.eval()`: Dropout 비활성화 (추론 시)</br>
> 모드 전환을 잊으면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">추론 결과가 매번 달라집니다</mark>.</br>
> </br>
> 시험 때마다 랜덤으로 팀원을 빼면, 남은 팀원들이 각자 독립적으로 문제를 풀 수 있어야 합니다.</br>
> → 특정 뉴런에 의존하지 않는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">견고한 학습</mark>이 됩니다.

</br>

## nn.Sequential
> 레이어를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">순차적으로 쌓는</mark> 간편한 방법입니다.

In [14]:
# TODO 6: nn.Sequential을 이용하여 784x256 → ReLU → Dropout → 256x128 → ReLU → 128x10 파이프라인을 구성해봅시다.
# 입력(784) -> 은닉층1(256) -> Dropout -> 은닉층2(128) -> 출력(10)
# Dropout은 과적합(Overfitting)을 막기 위해 학습 중 뉴런을 무작위로 끄는 기술입니다.
sequential_model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(p=0.5), # 50%의 뉴런을 무작위로 비활성화
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

print("--- Sequential 모델 구조 ---")
print(sequential_model)

--- Sequential 모델 구조 ---
Sequential(
  (0): Linear(in_features=784, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=256, out_features=128, bias=True)
  (4): ReLU()
  (5): Linear(in_features=128, out_features=10, bias=True)
)


In [15]:
# TODO 7: 1x784 랜덤 텐서를 생성하여 model을 호출해봅시다.
# 1장의 이미지(배치 크기=1), 784개의 픽셀(28x28 이미지 평탄화)을 가정한 더미 데이터
dummy_input = torch.randn(1, 784)

# 모델에 데이터를 통과시킵니다 (순전파, Forward Pass)
output = sequential_model(dummy_input)

print("\n--- 더미 데이터 추론 결과 ---")
print(f"입력 데이터 형태: {dummy_input.shape}")
print(f"출력 데이터 형태: {output.shape}") 
print(f"출력값 (Logits): {output.detach().numpy()}")


--- 더미 데이터 추론 결과 ---
입력 데이터 형태: torch.Size([1, 784])
출력 데이터 형태: torch.Size([1, 10])
출력값 (Logits): [[ 0.08877157 -0.08735391  0.0045329  -0.01450388  0.04813191 -0.04953031
   0.03672177  0.13512889 -0.25129205 -0.21063396]]


💡nn.Sequential vs nn.Module 커스텀
> 단순 순차 구조: `nn.Sequential`로 충분</br>
> 잔차 연결, 분기, 조건부 흐름: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">nn.Module 상속</mark>이 필요